In [1]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup, Comment
import time
from io import StringIO
import pickle 


In [2]:
ap_polls_url = "https://www.collegepollarchive.com/basketball/men/ap/seasons.cfm?appollid={}"

def get_ap_poll(pollid : int) -> pd.DataFrame:
    """Get the AP Poll for a given poll ID."""
    r = requests.get(ap_polls_url.format(pollid))
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    table = soup.find("table")
    date = soup.find("h2").text.split(" AP")[0].strip()
    df = pd.read_html(StringIO(str(table)))[0]
    df = df.iloc[:25]
    df = df.rename(columns={"Rank": "Current Rank", "Rank.2": "Previous Rank"})
    df["FPV"] = df["Team (FPV)"].str.extract(r"\((\d+)\)$").astype(float)
    df["Team"] = df["Team (FPV)"].str.replace(r"\s*\(.*\)$", "", regex=True)
    df = df.drop(["Rank.1", "Team (FPV)"], axis=1)

    df["FPV"] = df["FPV"].fillna(0)
    df["Record"] = df["Record"].fillna("0-0")
    df["W"] = df["Record"].str.extract(r"^(\d+)-").astype(int)
    df["L"] = df["Record"].str.extract(r"-(\d+)$").astype(int)
    df = df.drop("Record", axis=1)  

    return df, date

In [3]:
def get_all_season_ap_polls(ap_poll_ids : list[int]) -> pd.DataFrame:
    """Get all AP Polls for a given list of poll IDs."""
    all_polls = {}
    for pollid in ap_poll_ids:
        poll_df, date = get_ap_poll(pollid)
        poll_df["Poll ID"] = pollid
        poll_df["Date"] = date
        all_polls[date] = poll_df
    return all_polls

In [4]:
ap_poll = get_all_season_ap_polls([1310])["January 5, 2026"][["Team", "Current Rank"]].rename(columns={"Current Rank": "rank"})


In [5]:
ap_poll

,Team,rank
0,Arizona,1
1,Michigan,2
2,Iowa State,3
3,Connecticut,4
4,Purdue,5
5,Duke,6
6,Houston,7
7,Gonzaga,8
8,BYU,9
9,Nebraska,10


In [6]:
ap_poll.to_csv("../backend/data/current_ap.csv")

In [6]:
def get_game_box_score(game_link: str, date: str, gameID: str) -> pd.DataFrame:
    """Get the box score for a given game link."""
    box_score_url = f"https://www.sports-reference.com{game_link}"
    r = requests.get(box_score_url)
    if r.status_code != 200:
        print(f"Rate limited. Waiting {r.headers['Retry-After']} seconds.")
        time.sleep(int(r.headers["Retry-After"]))
        r = requests.get(box_score_url)
        
    soup = BeautifulSoup(r.text, "html.parser")
    comments = soup.find_all(string=lambda s: isinstance(s, Comment))
    line_score = None
    four_factors = None
    for c in comments:
        if '<table' in c and 'id="line-score"' in c:
            line_score = BeautifulSoup(c, "html.parser").find("table", id="line-score")
        elif '<table' in c and 'id="four-factors"' in c:
            four_factors = BeautifulSoup(c, "html.parser").find("table", id="four-factors")
    box_score_basics = soup.find_all("table", id=lambda x: x and x.startswith("box-score-basic-"))
    away_box_score_basic, home_box_score_basic = box_score_basics
    box_score_advanced = soup.find_all("table", id=lambda x: x and x.startswith("box-score-advanced-"))
    away_box_score_advanced, home_box_score_advanced = box_score_advanced

    line_score = pd.read_html(StringIO(str(line_score)))[0]
    four_factors = pd.read_html(StringIO(str(four_factors)))[0]
    away_box_score_basic = pd.read_html(StringIO(str(away_box_score_basic)))[0]
    home_box_score_basic = pd.read_html(StringIO(str(home_box_score_basic)))[0]
    away_box_score_advanced = pd.read_html(StringIO(str(away_box_score_advanced)))[0]
    home_box_score_advanced = pd.read_html(StringIO(str(home_box_score_advanced)))[0]

    box_score_dict =  {
        "GameID" : gameID,
        "Date" : date,
        "line_score": line_score,
        "four_factors": four_factors,
        "away_box_score_basic": away_box_score_basic,
        "home_box_score_basic": home_box_score_basic,
        "away_box_score_advanced": away_box_score_advanced,
        "home_box_score_advanced": home_box_score_advanced,
    }

    return box_score_dict

In [7]:
box_scores_url = "https://www.sports-reference.com/cbb/boxscores/index.cgi?month={}&day={}&year={}"

def get_day_games_data(month: int, day: int, year: int, throttle: int = 15, throttle_time: int = 60) -> pd.DataFrame:
    """Get all games data for a given day."""
    date = f"{year}-{month:02d}-{day:02d}"
    r = requests.get(box_scores_url.format(month, day, year))
    if r.status_code != 200:
        print(f"Rate limited. Waiting {r.headers['Retry-After']} seconds.")
        time.sleep(int(r.headers["Retry-After"]))
        r = requests.get(box_scores_url.format(month, day, year))
    soup = BeautifulSoup(r.text, "html.parser")
    summaries = soup.find("div", class_="game_summaries")
    men_games = summaries.find_all("div", class_="game_summary nohover gender-m")
    men_game_data_rows = []
    men_team_data_rows = []
    men_game_box_scores = []

    pages_visited = 0
    start_time = time.time()
    for game in men_games:
        teams = game.find_all("tr", class_=["winner", "loser"])
        if teams[0].find_all("td")[-2].text == "":
            continue
        for team in teams:
            location = "home" if teams.index(team) == 1 else "away"
            team_name = team.find("a").text
            pollrank = team.find("span", class_="pollrank")
            if pollrank:
                rank = pollrank.text.replace("(", "").replace(")", "")
            else: 
                rank = "NR"
            points = team.find_all("td")[-2].text
            team_data_row = {
                "GameID": f"{date}-{teams[0].find('a').text}-vs-{teams[1].find('a').text}-m",
                "Date": date,
                "Team": team_name,
                "Location": location,
                "Rank": rank,
                "Points": points,
                "Result": "W" if location == "home" and int(points) > int(teams[0].find_all("td")[-2].text) else 
                          "L" if location == "home" else
                          "W" if location == "away" and int(points) > int(teams[1].find_all("td")[-2].text) else 
                          "L"
            }
            men_team_data_rows.append(team_data_row)

        game_link = game.find("td", class_="right gamelink").find("a")["href"]

        gameID = f"{date}-{teams[0].find('a').text}-vs-{teams[1].find('a').text}-m"

        box_score_dict = get_game_box_score(game_link, date, gameID)

        men_game_box_scores.append(box_score_dict)

        game_data_row = {
            "GameID": gameID,
            "Date": date,
            "Home Team": teams[1].find("a").text,
            "Home Rank": team_data_row["Rank"] if team_data_row["Location"] == "home" else
                         teams[1].find("span", class_="pollrank").text.replace("(", "").replace(")", "") if teams[1].find("span", class_="pollrank") else "NR",
            "Home Points": teams[1].find_all("td")[-2].text,
            "Away Team": teams[0].find("a").text,
            "Away Rank": team_data_row["Rank"] if team_data_row["Location"] == "away" else
                         teams[0].find("span", class_="pollrank").text.replace("(", "").replace(")", "") if teams[0].find("span", class_="pollrank") else "NR",
            "Away Points": teams[0].find_all("td")[-2].text,
            "Winner": teams[1].find("a").text if int(teams[1].find_all("td")[-2].text) > int(teams[0].find_all("td")[-2].text) else teams[0].find("a").text
        }
        men_game_data_rows.append(game_data_row)

        print(f"Done with {game_data_row['Away Team']} @ {game_data_row['Home Team']} on {game_data_row['Date']}")
        
        pages_visited += 1
        if pages_visited >= throttle:
            sleep_time = max(throttle_time - (time.time() - start_time), 0)
            print(f"Sleeping for {sleep_time:.2f} seconds")
            time.sleep(sleep_time)
            start_time = time.time()
            pages_visited = 0

    men_games_df = pd.DataFrame(men_game_data_rows)
    men_teams_df = pd.DataFrame(men_team_data_rows)

    return men_games_df, men_teams_df, men_game_box_scores

In [8]:
def get_season_data(dates: list[tuple[int, int, int]], throttle: int = 15, throttle_time: int = 60):
    """Get all season data for given"""
    games_df = pd.DataFrame()
    teams_df = pd.DataFrame()
    box_scores = []
    for month, day, year in dates:
        men_games_df, men_teams_df, men_game_box_scores = get_day_games_data(month, day, year, throttle, throttle_time)
        games_df = pd.concat([games_df, men_games_df], ignore_index=True)
        teams_df = pd.concat([teams_df, men_teams_df], ignore_index=True)
        box_scores.extend(men_game_box_scores)
        print("Done with date:", f"{year}-{month:02d}-{day:02d}")
        print("Sleeping for 60 seconds before next date...")
        time.sleep(60)
    return games_df, teams_df, box_scores


In [9]:
def get_dates(start, end):
    """Generate a list of dates between start and end."""
    from datetime import datetime, timedelta
    start_date = datetime.strptime(start, "%Y-%m-%d")
    end_date = datetime.strptime(end, "%Y-%m-%d")
    delta = end_date - start_date
    dates = []
    for i in range(delta.days + 1):
        day = start_date + timedelta(days=i)
        dates.append((day.month, day.day, day.year))
    return dates

In [10]:
def format_line_score(line_score: pd.DataFrame) -> pd.DataFrame:
    """Format the line score dataframe."""
    data = {
        "Team" : line_score[("Scoring", "Unnamed: 0_level_1")],
        "H1" : line_score[("Scoring", "1")],
        "H2" : line_score[("Scoring", "2")],
        "T" : line_score[("Scoring", "T")],
    }
    if "OT" in line_score[("Scoring",)].columns:
        data["OT"] = line_score[("Scoring", "OT")]
    line_score = pd.DataFrame(data)
    return line_score

def format_four_factors(four_factors: pd.DataFrame) -> pd.DataFrame:
    """Format the four factors dataframe."""
    data = {
        "Team" : four_factors[("Unnamed: 0_level_0", "Unnamed: 0_level_1")],
        "Pace" : four_factors[("Unnamed: 1_level_0", "Pace")],
        "eFG%" : four_factors[("Four Factors", "eFG%")],
        "TOV%" : four_factors[("Four Factors", "TOV%")],
        "ORB%" : four_factors[("Four Factors", "ORB%")],
        "FT/FGA" : four_factors[("Four Factors", "FT/FGA")],
        "ORtg" : four_factors[("Unnamed: 6_level_0", "ORtg")],
    }
    four_factors = pd.DataFrame(data)
    return four_factors

def format_box_score_basic(box_score: dict, location: str) -> pd.DataFrame:
    """Format the box score basic dataframe."""
    gameID = box_score["GameID"]
    date = box_score["Date"]
    box_score_basic = box_score["away_box_score_basic"] if location == "away" else box_score["home_box_score_basic"]
    team = box_score["line_score"].iloc[0,0] if location == "away" else box_score["line_score"].iloc[1,0]
    opp_team = box_score["line_score"].iloc[1,0] if location == "away" else box_score["line_score"].iloc[0,0]
    result = box_score["line_score"].iloc[0, -1] > box_score["line_score"].iloc[1, -1] if location == "away" else box_score["line_score"].iloc[1, -1] > box_score["line_score"].iloc[0, -1]
    box = "W" if result else "L" 

    data = {
        "Player" : box_score_basic[("Unnamed: 0_level_0", "Starters")] if "Starters" in box_score_basic[("Unnamed: 0_level_0",)].columns else box_score_basic[("Unnamed: 0_level_0", "Player")],
        "GameID" : gameID,
        "Date" : date, 
        "Team" : [team] * len(box_score_basic),
        "Opponent" : [opp_team] * len(box_score_basic),
        "MP" : box_score_basic[("Basic Box Score Stats", "MP")],
        "FG" : box_score_basic[("Basic Box Score Stats", "FG")],
        "FGA" : box_score_basic[("Basic Box Score Stats", "FGA")],
        "FG%" : box_score_basic[("Basic Box Score Stats", "FG%")],
        "2P" : box_score_basic[("Basic Box Score Stats", "2P")],
        "2PA" : box_score_basic[("Basic Box Score Stats", "2PA")],
        "2P%" : box_score_basic[("Basic Box Score Stats", "2P%")],
        "3P" : box_score_basic[("Basic Box Score Stats", "3P")],
        "3PA" : box_score_basic[("Basic Box Score Stats", "3PA")],
        "3P%" : box_score_basic[("Basic Box Score Stats", "3P%")],
        "FT" : box_score_basic[("Basic Box Score Stats", "FT")],
        "FTA" : box_score_basic[("Basic Box Score Stats", "FTA")],
        "FT%" : box_score_basic[("Basic Box Score Stats", "FT%")],
        "ORB" : box_score_basic[("Basic Box Score Stats", "ORB")],
        "DRB" : box_score_basic[("Basic Box Score Stats", "DRB")],
        "TRB" : box_score_basic[("Basic Box Score Stats", "TRB")],
        "AST" : box_score_basic[("Basic Box Score Stats", "AST")],
        "STL" : box_score_basic[("Basic Box Score Stats", "STL")],
        "BLK" : box_score_basic[("Basic Box Score Stats", "BLK")],
        "TOV" : box_score_basic[("Basic Box Score Stats", "TOV")],
        "PF" : box_score_basic[("Basic Box Score Stats", "PF")],
        "PTS" : box_score_basic[("Basic Box Score Stats", "PTS")],
        "GmSc" : box_score_basic[("Basic Box Score Stats", "GmSc")],
        "Result" : [box] * len(box_score_basic)
    }

    role = ["Starter" for i in range(5)] + ["Reserve" for i in range(len(box_score_basic) - 5)]
    data["Role"] = role
    box_score_basic = pd.DataFrame(data)
    box_score_basic = box_score_basic.drop(5, axis=0)
    return box_score_basic

def format_box_score_advanced(box_score: dict, location: str) -> pd.DataFrame:
    """Format the box score advanced dataframe."""
    box_score_advanced = box_score["away_box_score_advanced"] if location == "away" else box_score["home_box_score_advanced"]
    team = box_score["line_score"].iloc[0,0] if location == "away" else box_score["line_score"].iloc[1,0]
    opp_team = box_score["line_score"].iloc[1,0] if location == "away" else box_score["line_score"].iloc[0,0]
    result = box_score["line_score"].iloc[0, -1] > box_score["line_score"].iloc[1, -1] if location == "away" else box_score["line_score"].iloc[1, -1] > box_score["line_score"].iloc[0, -1]
    box = "W" if result else "L"

    data = {
        "Player" : box_score_advanced[("Unnamed: 0_level_0", "Starters")] if "Starters" in box_score_advanced[("Unnamed: 0_level_0",)].columns else box_score_advanced[("Unnamed: 0_level_0", "Player")],
        "Team" : [team] * len(box_score_advanced),
        "Opponent" : [opp_team] * len(box_score_advanced),
        "MP" : box_score_advanced[("Advanced Box Score Stats", "MP")],
        "TS%" : box_score_advanced[("Advanced Box Score Stats", "TS%")],
        "eFG%" : box_score_advanced[("Advanced Box Score Stats", "eFG%")],
        "3PAr" : box_score_advanced[("Advanced Box Score Stats", "3PAr")],
        "FTr" : box_score_advanced[("Advanced Box Score Stats", "FTr")],
        "ORB%" : box_score_advanced[("Advanced Box Score Stats", "ORB%")],
        "DRB%" : box_score_advanced[("Advanced Box Score Stats", "DRB%")],
        "TRB%" : box_score_advanced[("Advanced Box Score Stats", "TRB%")],
        "AST%" : box_score_advanced[("Advanced Box Score Stats", "AST%")],
        "STL%" : box_score_advanced[("Advanced Box Score Stats", "STL%")],
        "BLK%" : box_score_advanced[("Advanced Box Score Stats", "BLK%")],
        "TOV%" : box_score_advanced[("Advanced Box Score Stats", "TOV%")],
        "USG%" : box_score_advanced[("Advanced Box Score Stats", "USG%")],
        "ORtg" : box_score_advanced[("Advanced Box Score Stats", "ORtg")],
        "DRtg" : box_score_advanced[("Advanced Box Score Stats", "DRtg")],
        "Result" : [box] * len(box_score_advanced)
    }

    role = ["Starter" for i in range(5)] + ["Reserve" for i in range(len(box_score_advanced) - 5)]
    data["Role"] = role

    if "BPM" in box_score_advanced[("Advanced Box Score Stats",)].columns:
        data["BPM"] = box_score_advanced[("Advanced Box Score Stats", "BPM")]
    else:
        data["BPM"] = [np.nan] * len(box_score_advanced)
        
    box_score_advanced = pd.DataFrame(data)
    box_score_advanced = box_score_advanced.drop(5, axis=0)
    return box_score_advanced

def format_box_score(box_score: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Format the box score dataframes."""
    line_score = format_line_score(box_score["line_score"])
    four_factors = format_four_factors(box_score["four_factors"])
    away_box_score_basic = format_box_score_basic(box_score, "away")
    home_box_score_basic = format_box_score_basic(box_score, "home")
    away_box_score_advanced = format_box_score_advanced(box_score, "away")
    home_box_score_advanced = format_box_score_advanced(box_score, "home")
    return (line_score, four_factors, away_box_score_basic, home_box_score_basic, away_box_score_advanced, home_box_score_advanced)

In [11]:
def create_player_games_df(box_scores: list[dict]) -> pd.DataFrame:
    """Create a dataframe of player games from box scores."""
    players_df = pd.DataFrame()
    for box in box_scores:
        box_score_home = box["home_box_score_basic"]
        box_score_away = box["away_box_score_basic"]
        players_df = pd.concat([players_df, box_score_home, box_score_away], ignore_index=True)
    return players_df


In [12]:
def get_team_totals_df(box_scores: list[dict]) -> pd.DataFrame:
    """Create a dataframe of team totals from box scores."""
    team_totals_df = pd.DataFrame()
    for box in box_scores:
        if "Player" not in box["home_box_score_basic"].columns:
            print(box["home_box_score_basic"].columns)
        home_box_score_basic = box["home_box_score_basic"]
        away_box_score_basic = box["away_box_score_basic"]
        if "Player" in box["home_box_score_basic"].columns:
            home_team_totals = home_box_score_basic[home_box_score_basic["Player"] == "School Totals"]
        else:
            home_team_totals = home_box_score_basic[home_box_score_basic["Starters"] == "School Totals"]
        away_team_totals = away_box_score_basic[away_box_score_basic["Player"] == "School Totals"]
        team_totals_df = pd.concat([team_totals_df, home_team_totals, away_team_totals], ignore_index=True)
    return team_totals_df



In [21]:
def update_data(date_start, date_end):
    dates = get_dates(date_start, date_end)
    games_df, teams_df, box_scores = get_season_data(dates)

    existing_games_df = pd.read_csv("games.csv").drop("Unnamed: 0", axis=1)
    existing_teams_df = pd.read_csv("teams.csv").drop("Unnamed: 0", axis=1)

    games_df = pd.concat([existing_games_df, games_df], axis=0)
    teams_df = pd.concat([existing_teams_df, teams_df], axis=0)

    for box in box_scores:
        (line_score, four_factors, away_box_score_basic, home_box_score_basic, away_box_score_advanced, home_box_score_advanced) = format_box_score(box)
        box["line_score"] = line_score
        box["four_factors"] = four_factors
        box["away_box_score_basic"] = away_box_score_basic
        box["home_box_score_basic"] = home_box_score_basic
        box["away_box_score_advanced"] = away_box_score_advanced
        box["home_box_score_advanced"] = home_box_score_advanced
    
    with open("box_scores.pkl", "rb") as file:
        existing_box_scores = pickle.load(file)

    existing_box_scores.extend(box_scores)
    players_box_df = create_player_games_df(existing_box_scores)
    team_totals_df = get_team_totals_df(existing_box_scores)
    team_totals_df = team_totals_df.drop(["Player", "Opponent", "Role", "GmSc", "GameID", "Date"], axis=1)
    team_totals_df["W"] = (team_totals_df["Result"] == "W").astype(int)
    team_totals_df["L"] = (team_totals_df["Result"] == "L").astype(int)

    team_totals_df = team_totals_df.drop(["Result"], axis=1)

    team_totals_df[[i for i in team_totals_df.columns if i != "Team"]] = team_totals_df[[i for i in team_totals_df.columns if i != "Team"]].astype("float")

    team_totals = team_totals_df.groupby("Team").agg({
        "MP" : "sum",
        "FG" : "sum",
        "FGA" : "sum",
        "FT" : "sum",
        "FTA" : "sum",
        "3P" : "sum",
        "3PA" : "sum",
        "ORB" : "sum",
        "DRB" : "sum",
        "TRB" : "sum",
        "AST" : "sum",
        "STL" : "sum",
        "BLK" : "sum",
        "TOV" : "sum",
        "PF" : "sum",
        "PTS" : "sum",
        "W" : "sum",
        "L" : "sum",
        "FG%" : "mean",
        "FT%" : "mean",
        "3P%" : "mean",
        "2P%" : "mean",
    })

    team_averages = team_totals_df.groupby("Team").agg({
        "MP" : "mean",
        "FG" : "mean",
        "FGA" : "mean",
        "FT" : "mean",
        "FTA" : "mean",
        "3P" : "mean",
        "3PA" : "mean",
        "ORB" : "mean",
        "DRB" : "mean",
        "TRB" : "mean",
        "AST" : "mean",
        "STL" : "mean",
        "BLK" : "mean",
        "TOV" : "mean",
        "PF" : "mean",
        "PTS" : "mean",
        "W" : "sum",
        "L" : "sum",
        "FG%" : "mean",
        "FT%" : "mean",
        "3P%" : "mean",
        "2P%" : "mean"
    })

    team_totals= team_totals.round(2)
    team_averages = team_averages.round(2)

    team_averages.sort_values(by="PTS", ascending=False)

    with open("box_scores_updated.pkl", "wb") as file:
        pickle.dump(existing_box_scores, file)
    
    players_box_df.to_csv("players_box.csv")
    games_df.to_csv("games.csv")
    teams_df.to_csv("teams.csv")
    team_totals.to_csv("team_totals.csv")
    team_averages.to_csv("team_averages.csv")

In [22]:
update_data("2025-11-22", "2025-11-22")

Done with LIU @ Illinois on 2025-11-22
Done with North Carolina Central @ Dayton on 2025-11-22
Done with Mercer @ Eastern Kentucky on 2025-11-22
Done with Ball State @ Indiana State on 2025-11-22
Done with Gardner-Webb @ Richmond on 2025-11-22
Done with VMI @ Stetson on 2025-11-22
Done with Coppin State @ VCU on 2025-11-22
Done with Western Carolina @ Lipscomb on 2025-11-22
Done with UT-Martin @ Prairie View on 2025-11-22
Done with Milwaukee @ Wichita State on 2025-11-22
Done with UCSB @ Nevada on 2025-11-22
Done with Wagner @ Georgetown on 2025-11-22
Done with NJIT @ Navy on 2025-11-22
Done with Harvard @ Boston University on 2025-11-22
Done with Fairfield @ Le Moyne on 2025-11-22
Sleeping for 52.88 seconds
Done with VTSU Lyndon @ New Haven on 2025-11-22
Done with Cleveland State @ Kent State on 2025-11-22
Done with Central Michigan @ Marquette on 2025-11-22
Done with UMass-Lowell @ St. Peter's on 2025-11-22
Done with Howard Payne @ Texas A&M-Corpus Christi on 2025-11-22
Done with Mar

/var/folders/p_/d5kqctzj6579dv531381_klh0000gn/T/ipykernel_67140/1283141863.py:39: PerformanceWarning: indexing past lexsort depth may impact performance.
  "Player" : box_score_basic[("Unnamed: 0_level_0", "Starters")] if "Starters" in box_score_basic[("Unnamed: 0_level_0",)].columns else box_score_basic[("Unnamed: 0_level_0", "Player")],
/var/folders/p_/d5kqctzj6579dv531381_klh0000gn/T/ipykernel_67140/1283141863.py:39: PerformanceWarning: indexing past lexsort depth may impact performance.
  "Player" : box_score_basic[("Unnamed: 0_level_0", "Starters")] if "Starters" in box_score_basic[("Unnamed: 0_level_0",)].columns else box_score_basic[("Unnamed: 0_level_0", "Player")],
/var/folders/p_/d5kqctzj6579dv531381_klh0000gn/T/ipykernel_67140/1283141863.py:85: PerformanceWarning: indexing past lexsort depth may impact performance.
  "Player" : box_score_advanced[("Unnamed: 0_level_0", "Starters")] if "Starters" in box_score_advanced[("Unnamed: 0_level_0",)].columns else box_score_advanced[